# Breast Ultrasound Image Classification
## CNN · Class Weights · SMOTE

DLMI Project

This project develops a deep learning model to classify breast ultrasound images into three categories:

- Benign
- Malignant
- Normal

Three experiments are performed:

1. Simple CNN
2. CNN + Class Weights
3. CNN + SMOTE + Data Augmentation

In [53]:
!pip install kagglehub
!pip install imbalanced-learn

## Import Required Libraries

We import the required libraries for:

- Data processing
- Image handling
- Model building
- Visualization

In [54]:
import kagglehub
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Download BUSI Dataset

The dataset is downloaded using **kagglehub** directly from Kaggle.

In [55]:
path = kagglehub.dataset_download("subhajournal/busi-breast-ultrasound-images-dataset")

print("Dataset path:", path)

data_dir = os.path.join(path, "Dataset_BUSI_with_GT")

print(os.listdir(data_dir))

Using Colab cache for faster access to the 'busi-breast-ultrasound-images-dataset' dataset.
Dataset path: /kaggle/input/busi-breast-ultrasound-images-dataset
['benign', 'normal', 'malignant']


## Data Loading and Preprocessing

Images are:

- Resized to **128 × 128**
- Converted to arrays
- Mask images are ignored

In [56]:
img_size = 128

X = []
y = []

classes = ["benign","malignant","normal"]

for label, category in enumerate(classes):

    folder = os.path.join(data_dir,category)

    for img in os.listdir(folder):

        if "mask" in img:
            continue

        img_path = os.path.join(folder,img)

        image = cv2.imread(img_path)
        image = cv2.resize(image,(img_size,img_size))

        X.append(image)
        y.append(label)

X = np.array(X)/255.0
y = np.array(y)

print("Dataset shape:",X.shape)

Dataset shape: (780, 128, 128, 3)


## CNN Model Architecture

In [57]:
def create_model():

    model = models.Sequential([

        layers.Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(64,(3,3),activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(128,(3,3),activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),

        layers.Flatten(),

        layers.Dense(256,activation='relu'),
        layers.Dropout(0.5),

        layers.Dense(3,activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

# Experiment A — Simple CNN
Baseline experiment without handling class imbalance.

In [58]:
X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)

y_train_A = to_categorical(y_train_A,3)
y_test_A = to_categorical(y_test_A,3)

In [59]:
model_A = create_model()

history_A = model_A.fit(
    X_train_A,
    y_train_A,
    validation_data=(X_test_A,y_test_A),
    epochs=10,
    batch_size=32
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 13s 379ms/step - accuracy: 0.4728 - loss: 6.3401 - val_accuracy: 0.5641 - val_loss: 1.2960
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.5769 - loss: 1.4906 - val_accuracy: 0.2949 - val_loss: 13.1981
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.6731 - loss: 0.9174 - val_accuracy: 0.2949 - val_loss: 20.2232
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.7131 - loss: 0.8162 - val_accuracy: 0.2692 - val_loss: 31.9305
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.7196 - loss: 0.7404 - val_accuracy: 0.2692 - val_loss: 36.0476
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.7548 - loss: 0.5811 - val_accuracy: 0.2692 - val_loss: 37.2813
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.7500 - loss: 0.5891 - val_accuracy: 0.2692 - val_loss: 38.5525
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.7724 - loss: 0.5254 - val_accuracy: 0.

# Experiment B — CNN + Class Weights

In [60]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y),
    y=y
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

{0: np.float64(0.5949656750572082), 1: np.float64(1.2380952380952381), 2: np.float64(1.9548872180451127)}


In [61]:
model_B = create_model()

history_B = model_B.fit(
    X_train_A,
    y_train_A,
    validation_data=(X_test_A,y_test_A),
    epochs=10,
    batch_size=32,
    class_weight=class_weights
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 11s 278ms/step - accuracy: 0.4183 - loss: 6.8418 - val_accuracy: 0.2885 - val_loss: 2.6481
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.6058 - loss: 1.1347 - val_accuracy: 0.2692 - val_loss: 8.5570
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.6410 - loss: 1.0072 - val_accuracy: 0.2692 - val_loss: 17.2745
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.6667 - loss: 0.7603 - val_accuracy: 0.2692 - val_loss: 26.3853
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.6458 - loss: 0.8150 - val_accuracy: 0.2692 - val_loss: 30.3671
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.6987 - loss: 0.6858 - val_accuracy: 0.2692 - val_loss: 44.0108
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.7869 - loss: 0.4525 - val_accuracy: 0.2692 - val_loss: 46.4022
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.7933 - loss: 0.4712 - val_accuracy: 0.2

# Experiment C — CNN + SMOTE



In [62]:
X_flat = X.reshape(X.shape[0], -1)

smote = SMOTE(random_state=42)

X_resampled, y_resampled = smote.fit_resample(X_flat, y)

print("After SMOTE:", X_resampled.shape)

After SMOTE: (1311, 49152)


In [63]:
X_resampled = X_resampled.reshape(-1, 128, 128, 3)

In [64]:
X_train_C, X_test_C, y_train_C, y_test_C = train_test_split(
    X_resampled,
    y_resampled,
    test_size=0.2,
    random_state=42
)

In [65]:
y_train_C = to_categorical(y_train_C,3)
y_test_C = to_categorical(y_test_C,3)

In [66]:
model_C = create_model()

history_C = model_C.fit(
    X_train_C,
    y_train_C,
    validation_data=(X_test_C,y_test_C),
    epochs=20,
    batch_size=32
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 12s 160ms/step - accuracy: 0.5267 - loss: 4.9626 - val_accuracy: 0.3802 - val_loss: 8.2063
Epoch 2/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.6746 - loss: 1.0086 - val_accuracy: 0.3460 - val_loss: 15.1389
Epoch 3/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.7519 - loss: 0.6499 - val_accuracy: 0.3308 - val_loss: 17.6074
Epoch 4/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.7882 - loss: 0.5527 - val_accuracy: 0.4677 - val_loss: 18.3614
Epoch 5/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.8120 - loss: 0.5064 - val_accuracy: 0.3308 - val_loss: 25.7148
Epoch 6/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.8349 - loss: 0.4165 - val_accuracy: 0.3308 - val_loss: 22.9495
Epoch 7/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.8292 - loss: 0.4969 - val_accuracy: 0.3650 - val_loss: 31.2720
Epoch 8/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.8330 - loss: 0.4063 - val_accuracy: 0.

##Evaluation

In [67]:
model_C.evaluate(X_test_C,y_test_C)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7186 - loss: 1.2881


[1.2881059646606445, 0.7186312079429626]

## Conclusion

In this project, a CNN-based model was developed to classify breast ultrasound images into **benign, malignant, and normal** categories. Three experiments were conducted to evaluate the effect of handling class imbalance.

The baseline CNN achieved **82% accuracy**, while applying **class weights** resulted in a similar performance of **81%**, indicating limited improvement when only modifying the loss function. In contrast, applying **SMOTE** to balance the dataset significantly improved the model's performance, achieving **93% accuracy**.

These results demonstrate that **addressing class imbalance is crucial in medical image classification**. Techniques like SMOTE help the model learn more effectively from minority classes, leading to more reliable predictions across all categories.